# 蒙特卡洛模拟 - 定增收益预测

## 分析目标
使用蒙特卡洛方法模拟定增项目的收益分布，包括：
- 股价路径模拟（几何布朗运动）
- 定增收益分布
- 盈利/亏损概率分析
- 收益率区间预测

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from utils.analysis_tools import PrivatePlacementRiskAnalyzer

# 直接配置中文字体（适用于vnpy等虚拟环境）
from utils.direct_font_config import setup
setup()

# 获取字体属性（用于绘图时设置字体）
from utils.font_manager import get_font_prop
font_prop = get_font_prop()

%matplotlib inline
sns.set_style('whitegrid')

print('✅ 蒙特卡洛模拟模块加载成功')

## 1. 参数设置与模拟器初始化

In [ ]:
# 定增项目参数
PROJECT_PARAMS = {
    'issue_price': 20.0,
    'issue_shares': 5000000,
    'lockup_period': 12,  # 12个月锁定期
    'current_price': 18.5,
    'risk_free_rate': 0.03
}

# 蒙特卡洛模拟参数
MC_PARAMS = {
    'n_simulations': 10000,    # 模拟次数
    'volatility': 0.35,        # 年化波动率
    'drift': 0.08,              # 年化漂移率（预期收益率）
    'time_steps': 252,         # 交易天数
    'seed': 42                  # 随机种子（可复现）
}

# 创建分析器
analyzer = PrivatePlacementRiskAnalyzer(**PROJECT_PARAMS)

print(f"项目参数:")
print(f"  发行价格: {PROJECT_PARAMS['issue_price']} 元/股")
print(f"  当前价格: {PROJECT_PARAMS['current_price']} 元/股")
print(f"  锁定期: {PROJECT_PARAMS['lockup_period']} 个月")
print(f"\n模拟参数:")
print(f"  模拟次数: {MC_PARAMS['n_simulations']:,}")
print(f"  波动率: {MC_PARAMS['volatility']*100:.0f}%")
print(f"  漂移率: {MC_PARAMS['drift']*100:.0f}%")

## 2. 运行蒙特卡洛模拟

In [ ]:
# 运行模拟
print("开始蒙特卡洛模拟，请稍候...")

sim_results = analyzer.monte_carlo_simulation(
    n_simulations=MC_PARAMS['n_simulations'],
    time_steps=MC_PARAMS['time_steps'],
    volatility=MC_PARAMS['volatility'],
    drift=MC_PARAMS['drift'],
    seed=MC_PARAMS['seed']
)

print(f"✅ 模拟完成！生成 {MC_PARAMS['n_simulations']:,} 条路径")
print(f"\n模拟结果形状: {sim_results.shape}")

## 3. 模拟路径可视化

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# 绘制模拟路径
for i in range(n_paths_to_plot):
    path = sim_results.iloc[random_paths[i], :days_to_plot+1].values
    alpha = 0.1
    color = 'green' if path[-1] >= PROJECT_PARAMS['issue_price'] else 'red'
    ax.plot(range(days_to_plot+1), path, color=color, alpha=alpha, linewidth=0.5)

# 绘制关键水平线
ax.axhline(y=PROJECT_PARAMS['issue_price'], color='blue', linestyle='--', 
        linewidth=2, label=f"发行价格 ({PROJECT_PARAMS['issue_price']}元)")
ax.axhline(y=PROJECT_PARAMS['current_price'], color='orange', linestyle='--', 
        linewidth=2, label=f"当前价格 ({PROJECT_PARAMS['current_price']}元)")

ax.set_xlabel('天数', fontsize=12, fontproperties=font_prop)
ax.set_ylabel('股价 (元)', fontsize=12, fontproperties=font_prop)
ax.set_title(f'蒙特卡洛模拟 - 前{n_paths_to_plot}条路径', fontsize=14, fontweight='bold', fontproperties=font_prop)
ax.legend(loc='upper left', prop=font_prop)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. 收益分布分析

In [ ]:
# 提取锁定期末的价格
final_prices = sim_results.iloc[:, lockup_days].values

# 计算收益率
returns = (final_prices - PROJECT_PARAMS['issue_price']) / PROJECT_PARAMS['issue_price']

# 计算年化收益率
annualized_returns = (1 + returns) ** (12 / PROJECT_PARAMS['lockup_period']) - 1

# 统计摘要
print('\n=== 收益分布统计 ===')
print(f"平均收益率: {annualized_returns.mean()*100:.2f}%")
print(f"中位数收益率: {np.median(annualized_returns)*100:.2f}%")
print(f"标准差: {annualized_returns.std()*100:.2f}%")
print(f"\n盈利概率: {(returns > 0).sum() / len(returns) * 100:.2f}%")
print(f"破发概率: {(returns <= 0).sum() / len(returns) * 100:.2f}%")
print(f"\n最佳情况收益率: {annualized_returns.max()*100:.2f}%")
print(f"最差情况收益率: {annualized_returns.min()*100:.2f}%")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 简单收益率分布
ax1.hist(annualized_returns * 100, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax1.axvline(x=0, color='red', linestyle='--', linewidth=2, label='盈亏平衡')
ax1.axvline(x=np.mean(annualized_returns)*100, color='green', linestyle='--', 
         linewidth=2, label=f"平均值 ({np.mean(annualized_returns)*100:.2f}%)")
ax1.set_xlabel('年化收益率 (%)', fontsize=12, fontproperties=font_prop)
ax1.set_ylabel('频数', fontsize=12, fontproperties=font_prop)
ax1.set_title('定增收益率分布', fontsize=14, fontweight='bold', fontproperties=font_prop)
ax1.legend(prop=font_prop)
ax1.grid(True, alpha=0.3)

# 累积分布函数
sorted_returns = np.sort(annualized_returns)
cumulative = np.arange(1, len(sorted_returns) + 1) / len(sorted_returns)
ax2.plot(sorted_returns * 100, cumulative * 100, linewidth=2, color='darkblue')
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='盈亏平衡')
ax2.set_xlabel('年化收益率 (%)', fontsize=12, fontproperties=font_prop)
ax2.set_ylabel('累积概率 (%)', fontsize=12, fontproperties=font_prop)
ax2.set_title('收益率累积分布', fontsize=14, fontweight='bold', fontproperties=font_prop)
ax2.legend(prop=font_prop)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 收益率区间预测

In [ ]:
# 计算不同置信水平下的收益率区间
confidence_levels = [0.50, 0.68, 0.80, 0.90, 0.95]

print('\n=== 收益率区间预测 ===')
print(f"{'置信水平':<12} {'收益率区间':<30} {'最差情况':<15} {'最好情况':<15}")
print('-'*72)

for cl in confidence_levels:
    lower = np.percentile(annualized_returns, (1-cl)/2 * 100)
    upper = np.percentile(annualized_returns, (1+cl)/2 * 100)
    worst = np.percentile(annualized_returns, (1-cl) * 100)
    best = np.percentile(annualized_returns, cl * 100)
    
    print(f"{int(cl*100):>3}%:        [{lower*100:>7.2f}%, {upper*100:>7.2f}%]      {worst*100:>7.2f}%      {best*100:>7.2f}%")

## 6. 不同情景分析

In [ ]:
# 可视化
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# 1. 平均收益率 vs 波动率
ax1.plot(df_scenarios['volatility']*100, df_scenarios['mean_return']*100, 'o-', color='blue')
ax1.set_xlabel('波动率 (%)', fontproperties=font_prop)
ax1.set_ylabel('平均年化收益率 (%)', fontproperties=font_prop)
ax1.set_title('波动率 vs 收益率', fontproperties=font_prop)
ax1.grid(True, alpha=0.3)

# 2. 盈利概率 vs 波动率
ax2.plot(df_scenarios['volatility']*100, df_scenarios['profit_prob']*100, 's-', color='green', markersize=10)
ax2.set_xlabel('波动率 (%)', fontproperties=font_prop)
ax2.set_ylabel('盈利概率 (%)', fontproperties=font_prop)
ax2.set_title('波动率 vs 盈利概率', fontproperties=font_prop)
ax2.grid(True, alpha=0.3)

# 3. 5%分位数 vs 波动率
ax3.plot(df_scenarios['volatility']*100, df_scenarios['percentile_5']*100, '^-', color='red')
ax3.set_xlabel('波动率 (%)', fontproperties=font_prop)
ax3.set_ylabel('5%分位数收益率 (%)', fontproperties=font_prop)
ax3.set_title('波动率 vs 下行风险 (5% VaR)', fontproperties=font_prop)
ax3.grid(True, alpha=0.3)

# 4. 收益区间 (5%-95%)
ax4.fill_between(df_scenarios['volatility']*100, 
                 df_scenarios['percentile_5']*100, 
                 df_scenarios['percentile_95']*100, alpha=0.3, label='90%置信区间')
ax4.plot(df_scenarios['volatility']*100, df_scenarios['mean_return']*100, 'o-', label='平均值')
ax4.set_xlabel('波动率 (%)', fontproperties=font_prop)
ax4.set_ylabel('年化收益率 (%)', fontproperties=font_prop)
ax4.set_title('收益率区间', fontproperties=font_prop)
ax4.legend(prop=font_prop)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 风险调整后收益

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
metrics = ['夏普比率', '收益风险比', '索提诺比率']
values = [sharpe, reward_to_risk, sortino]
colors = ['steelblue', 'coral', 'mediumseagreen']

bars = ax.bar(metrics, values, color=colors, alpha=0.7)
ax.set_ylabel('数值', fontsize=12, fontproperties=font_prop)
ax.set_title('风险调整后收益指标', fontsize=14, fontweight='bold', fontproperties=font_prop)
ax.grid(True, axis='y', alpha=0.3)

# 添加数值标签
for bar, value in zip(bars, values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 8. 蒙特卡洛模拟结论

In [ ]:
print('\n' + '='*60)
print('蒙特卡洛模拟结论')
print('='*60)

print(f"\n📊 模拟配置:")
print(f"   模拟次数: {MC_PARAMS['n_simulations']:,}")
print(f"   假设波动率: {MC_PARAMS['volatility']*100:.0f}%")
print(f"   假设漂移率: {MC_PARAMS['drift']*100:.0f}%")

print(f"\n📈 收益预测:")
print(f"   预期年化收益率: {annualized_returns.mean()*100:.2f}%")
print(f"   中位数收益率: {np.median(annualized_returns)*100:.2f}%")
print(f"   90%置信区间: [{np.percentile(annualized_returns, 5)*100:.2f}%, {np.percentile(annualized_returns, 95)*100:.2f}%]")

print(f"\n⚠️ 风险评估:")
print(f"   盈利概率: {(returns > 0).sum() / len(returns) * 100:.1f}%")
print(f"   破发概率: {(returns <= 0).sum() / len(returns) * 100:.1f}%")
print(f"   夏普比率: {sharpe:.3f}")

# 风险等级判断
profit_prob = (returns > 0).sum() / len(returns)
if profit_prob >= 0.7:
    risk_level = "低风险"
    color = "🟢"
elif profit_prob >= 0.5:
    risk_level = "中等风险"
    color = "🟡"
else:
    risk_level = "高风险"
    color = "🔴"

print(f"\n{color} 综合风险评级: {risk_level}")